# 4. Purchasing problem: basic model

Lecture: *Week 5 - Mixed Integer Programming*, PDF pages 21–22 (*Purchasing Problem*, *Basic Model*).

**The problem.** Order five items with a budget of £160 and at most 10 kg of each, to maximise the expected profit:

| Item | 1 | 2 | 3 | 4 | 5 |
| --- | --- | --- | --- | --- | --- |
| Cost (£ per kg) | 10 | 12 | 9 | 5 | 13 |
| Expected profit (£ per kg) | 5 | 3 | 4 | 3 | 3 |

maximise $\sum_{i=1}^{5} P_i x_i$ subject to $\sum_{i=1}^{5} c_i x_i \le 160$, $0 \le x_i \le 10$, where $x_i$ is
the amount of item $i$ (kg, any real number). In the code, items 1–5 are `x[0]` … `x[4]`. This is an ordinary LP;
demos 5–7 add binary variables to it.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Expected profit **84.44**: 10 kg of item 1, 1.11 kg of item 3 and 10 kg of item 4; the whole
£160 is spent.

*Original script: `5 Source Files/Python/purchasing_lecture.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

The data from the table above.

In [ ]:
# Define cost and profit as lists
cost = [10, 12, 9, 5, 13]  # c_i: cost of items 1-5 (£ per kg)
profit = [5, 3, 4, 3, 3]   # P_i: expected profit of items 1-5 (£ per kg)

A *concrete model* is the container for the problem.

In [ ]:
# Create a concrete model
model = ConcreteModel()

Five real variables, each between 0 and 10 kg (`bounds=(0, 10)`).

In [ ]:
# Define the decision variables with upper bounds: 0 <= x[i] <= 10 kg of item i+1
model.x = Var(range(5), within=NonNegativeReals, bounds=(0, 10))

The objective: total expected profit, maximised.

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(profit[i] * model.x[i] for i in range(5)), sense=maximize)

The budget: the total cost must not exceed £160.

In [ ]:
# Define the inequality constraint: budget, sum of c_i x_i <= 160
model.cost_max = Constraint(expr=sum(cost[i] * model.x[i] for i in range(5)) <= 160)

HiGHS is the solver; unlike GLPK it runs in the browser.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

`pprint()` prints the model with the data filled in: check it against the problem above.

In [ ]:
# Display the model
model.pprint()

Solve. In the browser HiGHS prints its log here even with `tee=False` (on a desktop it stays quiet); the *Solving report* at the end should say *Status Optimal*.

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

`display()` prints the amounts, the profit, and the money spent (`Body` of `cost_max`).

In [ ]:
# Display the results
model.display()

## What do you notice?

Item 3 is ordered in a very small amount (1.11 kg). Why exactly that amount, and why could it be a problem?

<details><summary>Answer</summary>

Rank the items by profit per pound: item 4 (0.60), item 1 (0.50), item 3 (0.44), item 2 (0.25), item 5 (0.23).
The LP fills the best ones up to 10 kg (item 4: £50, item 1: £100) and spends the last £10 on item 3:
$10 / 9 = 1.11$ kg. A supplier may not accept such a small order. Demo 5 adds a minimum order of 6 kg, which
needs binary variables (PDF page 23).
</details>